In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
cust=pd.read_csv("customers.csv")
loan=pd.read_csv("loans.csv")
trn=pd.read_csv("transactions.csv")
cust.head()
cust.shape
cust.info()
cust.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer_ID      10000 non-null  str    
 1   Customer_Name    10000 non-null  str    
 2   Age              10000 non-null  int64  
 3   Gender           10000 non-null  str    
 4   City             10000 non-null  str    
 5   State            10000 non-null  str    
 6   Employment_Type  10000 non-null  str    
 7   Annual_Income    10000 non-null  float64
 8   Credit_Score     10000 non-null  int64  
 9   Account_Type     10000 non-null  str    
 10  Account_Balance  10000 non-null  float64
 11  Customer_Since   10000 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 937.6 KB


,Age,Annual_Income,Credit_Score,Account_Balance
count,10000.0000,10000.000000,10000.000000,10000.000000
mean,36.1236,63707.913570,705.576100,75322.279674
std,9.7151,38376.631974,63.568405,52751.528343
min,18.0000,12000.000000,426.000000,500.000000
25%,29.0000,37095.465000,662.000000,40075.100000
50%,36.0000,54635.390000,706.000000,63758.855000
75%,43.0000,80030.135000,750.000000,96620.367500
max,70.0000,427366.310000,850.000000,563036.930000


In [7]:
cust.isnull().sum()

Customer_ID        0
Customer_Name      0
Age                0
Gender             0
City               0
State              0
Employment_Type    0
Annual_Income      0
Credit_Score       0
Account_Type       0
Account_Balance    0
Customer_Since     0
dtype: int64

In [9]:
cust.duplicated().sum()

np.int64(0)

In [11]:
cust["Customer_ID"].nunique() #Total customers

10000

In [13]:
cust["Customer_ID"].count() #Total Customers

np.int64(10000)

In [15]:
cust["Annual_Income"].mean() #avarage income

np.float64(63707.913570000004)

In [17]:
cust["Credit_Score"].mean() #avarage cradit score

np.float64(705.5761)

In [19]:
cust.groupby("City")["Customer_ID"].count() #Customer by city

City
Ahmedabad      657
Bengaluru      617
Bhubaneswar    650
Chandigarh     619
Chennai        709
Delhi          686
Guwahati       677
Hyderabad      601
Indore         673
Jaipur         706
Kochi          667
Kolkata        656
Lucknow        701
Mumbai         728
Pune           653
Name: Customer_ID, dtype: int64

In [21]:
loan.groupby("Loan_ID").agg(
    Loan_Count=("Loan_ID","count"),
    Total_Loan_Amount=("Loan_Amount","sum"),
    Average_Interest=("Interest_Rate","mean"),
    Oustanding=("Outstanding_Amount","sum")
)

,Loan_Count,Total_Loan_Amount,Average_Interest,Oustanding
Loan_ID,,,,
L200000,1,203543.92,14.49,0.00
L200001,1,149331.03,11.71,133769.57
L200002,1,78910.61,16.88,0.00
L200003,1,54246.21,11.36,45293.62
L200004,1,73378.02,10.38,0.00
...,...,...,...,...
L214995,1,159708.45,5.50,24463.73
L214996,1,75098.51,10.24,29748.59
L214997,1,43366.00,16.11,29144.90


In [24]:
#Customer segmentation
def assign_segment(row):
    if row["Annual_Income"] >= 200000 and row["Account_Balance"] >= 200000:
        return "Premium"
    elif row["Annual_Income"] >= 100000 or row["Account_Balance"] >= 100000:
        return "High Value"
    elif row["Annual_Income"] >= 50000 or row["Account_Balance"] >= 50000:
        return "Mass Affluent"
    else:
        return "Standard"


# Apply the function across each row
cust["Customer_Segment"] = cust.apply(assign_segment, axis=1)

In [25]:
# anoter way
cust["Customer_Segment"] = np.select(
    [
        (cust["Annual_Income"] >= 200000) &
        (cust["Account_Balance"] >= 200000),

        (cust["Annual_Income"] >= 100000) |
        (cust["Account_Balance"] >= 100000),

        (cust["Annual_Income"] >= 50000) |
        (cust["Account_Balance"] >= 50000)
    ],
    [
        "Premium",
        "High Value",
        "Mass Affluent"
    ],
    default="Standard"
)

In [26]:
cust["Customer_Segment"].value_counts()

Customer_Segment
Mass Affluent    4743
High Value       2688
Standard         2510
Premium            59
Name: count, dtype: int64

In [27]:
#Find high-risk customers
high_risk=cust[cust["Credit_Score"]<600]
print(high_risk)

     Customer_ID Customer_Name  Age  Gender         City          State  \
8        C100008    Neha Singh   31    Male      Kolkata    West Bengal   
18       C100018    Riya Gupta   27  Female     Guwahati          Assam   
50       C100050    Neha Gupta   39  Female         Pune    Maharashtra   
56       C100056   Arjun Singh   28  Female        Delhi          Delhi   
85       C100085  Rahul Sharma   31    Male    Hyderabad      Telangana   
...          ...           ...  ...     ...          ...            ...   
9843     C109843   Rahul Patel   23    Male      Lucknow  Uttar Pradesh   
9844     C109844    Ankit Bose   32   Other       Mumbai    Maharashtra   
9858     C109858   Arjun Verma   42  Female  Bhubaneswar         Odisha   
9892     C109892  Ananya Verma   32  Female       Jaipur      Rajasthan   
9941     C109941   Kavya Patel   54    Male      Kolkata    West Bengal   

     Employment_Type  Annual_Income  Credit_Score     Account_Type  \
8           Salaried       48

In [28]:
high_risk_loans=high_risk.merge(loan,on="Customer_ID",how="inner")
print(high_risk_loans)

    Customer_ID Customer_Name  Age  Gender         City        State  \
0       C100050    Neha Gupta   39  Female         Pune  Maharashtra   
1       C100050    Neha Gupta   39  Female         Pune  Maharashtra   
2       C100056   Arjun Singh   28  Female        Delhi        Delhi   
3       C100056   Arjun Singh   28  Female        Delhi        Delhi   
4       C100085  Rahul Sharma   31    Male    Hyderabad    Telangana   
..          ...           ...  ...     ...          ...          ...   
720     C109844    Ankit Bose   32   Other       Mumbai  Maharashtra   
721     C109858   Arjun Verma   42  Female  Bhubaneswar       Odisha   
722     C109892  Ananya Verma   32  Female       Jaipur    Rajasthan   
723     C109941   Kavya Patel   54    Male      Kolkata  West Bengal   
724     C109941   Kavya Patel   54    Male      Kolkata  West Bengal   

    Employment_Type  Annual_Income  Credit_Score     Account_Type  ...  \
0          Salaried       18985.59           579          Sav

In [29]:
#Monthly transaction analysis
trn["Transaction_Date"] = pd.to_datetime(
    trn["Transaction_Date"])

In [30]:
trn["YearMonth"] =(trn["Transaction_Date"]
    .dt.to_period("M"))

In [31]:
monthly = (
    trn[
        trn["Transaction_Status"] == "Success"
    ]
    .groupby("YearMonth")["Amount"]
    .sum()
)

In [32]:
monthly.pct_change() * 100

YearMonth
2025-01          NaN
2025-02   -11.280019
2025-03    17.073817
2025-04    -8.971445
2025-05    13.475511
2025-06   -10.127964
2025-07     7.233407
2025-08    -6.052924
2025-09    -1.383286
2025-10     1.363656
2025-11    -4.028120
2025-12    19.229634
2026-01   -16.701134
2026-02    -4.969961
2026-03     8.508114
2026-04    -1.165198
2026-05    12.572631
2026-06    -6.848450
2026-07     5.738403
2026-08    -4.277642
2026-09   -14.226999
Freq: M, Name: Amount, dtype: float64

In [35]:
trn.to_csv("transactions1.csv",index=False)

In [38]:
import urllib.parse
from sqlalchemy import create_engine

username = "root"
# This safely encodes characters like @, #, $, or /
password = urllib.parse.quote_plus("Satya123@")  
host = "localhost"
database_name = "Bank_analysis"

# Rebuild the engine connection
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database_name}")


In [39]:
trn.to_sql(
    name="transactions1",
    con=engine,
    if_exists="replace",
    index=False
) 

50000